In [1]:
import os, glob

RUN_DIR = "../fine_tune/clipseg_lora_cloudsen"

print("RUN_DIR exists:", os.path.isdir(RUN_DIR))
paths = sorted(glob.glob(os.path.join(RUN_DIR, "checkpoint-*", "trainer_state.json")))

print("trainer_state.json files found:", len(paths))
for p in paths[:5]:
    print(" -", p)
if len(paths) > 5:
    print(" ...")
    print(" -", paths[-1])


RUN_DIR exists: True
trainer_state.json files found: 3
 - ../fine_tune/clipseg_lora_cloudsen/checkpoint-10000/trainer_state.json
 - ../fine_tune/clipseg_lora_cloudsen/checkpoint-8000/trainer_state.json
 - ../fine_tune/clipseg_lora_cloudsen/checkpoint-9000/trainer_state.json


In [2]:
import json, re
import pandas as pd

def step_from_ckpt_path(p: str) -> int:
    m = re.search(r"checkpoint-(\d+)", p)
    return int(m.group(1)) if m else -1

eval_rows = []

for p in paths:
    ckpt_step = step_from_ckpt_path(p)
    with open(p, "r") as f:
        st = json.load(f)
    for h in st.get("log_history", []):
        if "eval_loss" in h:
            row = dict(h)
            row["_ckpt_step_folder"] = ckpt_step
            row["_trainer_state_path"] = p
            eval_rows.append(row)

eval_df = pd.DataFrame(eval_rows)

print("Eval entries found:", len(eval_df))
if len(eval_df):
    print("Eval columns:", list(eval_df.columns))
    eval_df.head(5)


Eval entries found: 27
Eval columns: ['epoch', 'eval_dice', 'eval_iou', 'eval_loss', 'eval_runtime', 'eval_samples_per_second', 'eval_steps_per_second', 'step', '_ckpt_step_folder', '_trainer_state_path']


In [8]:
if len(eval_df) == 0:
    raise RuntimeError("No eval logs found. That means evaluation didn't run (or eval_loss never got logged).")

eval_df["step"] = pd.to_numeric(eval_df["step"], errors="coerce")
eval_df["epoch"] = pd.to_numeric(eval_df["epoch"], errors="coerce")

eval_df = (eval_df
           .dropna(subset=["step"])
           .sort_values("step")
           .drop_duplicates(subset=["step"], keep="last")
           .reset_index(drop=True))

print("Unique eval steps:", len(eval_df))
eval_df.tail(5)


Unique eval steps: 10


,epoch,eval_dice,eval_iou,eval_loss,eval_runtime,eval_samples_per_second,eval_steps_per_second,step,_source_ckpt_folder,_source_file
5,2.83,0.482542,0.631048,0.693842,37.8963,56.470,28.235,6000,9000,../fine_tune/clipseg_lora_cloudsen/checkpoint-...
6,3.30,0.482975,0.636088,0.694226,43.2903,49.434,24.717,7000,9000,../fine_tune/clipseg_lora_cloudsen/checkpoint-...
7,3.77,0.484493,0.651088,0.688570,39.4920,54.188,27.094,8000,8000,../fine_tune/clipseg_lora_cloudsen/checkpoint-...
8,4.24,0.485676,0.635386,0.685700,36.6376,58.410,29.205,9000,9000,../fine_tune/clipseg_lora_cloudsen/checkpoint-...
9,4.71,0.486639,0.636633,0.684879,39.5134,54.159,27.079,10000,10000,../fine_tune/clipseg_lora_cloudsen/checkpoint-...


In [ ]:
import os

def guess_checkpoint_path(run_dir: str, step: int) -> str:
    cand = os.path.join(run_dir, f"checkpoint-{int(step)}")
    return cand if os.path.isdir(cand) else "(checkpoint folder not found for this step)"

def best_by(col: str, mode: str = "max"):
    if col not in eval_df.columns:
        return None
    s = pd.to_numeric(eval_df[col], errors="coerce")
    tmp = eval_df.copy()
    tmp[col] = s
    tmp = tmp.dropna(subset=[col])
    if len(tmp) == 0:
        return None
    idx = tmp[col].idxmax() if mode == "max" else tmp[col].idxmin()
    return tmp.loc[idx]

best = None
metric_used = None

for col, mode in [("eval_iou", "max"), ("eval_dice", "max"), ("eval_loss", "min")]:
    r = best_by(col, mode)
    if r is not None:
        best = r
        metric_used = (col, mode)
        break

print("Metric used:", metric_used)
print("\n=== BEST CHECKPOINT ===")
print("step :", int(best['step']))
print("epoch:", float(best['epoch']) if pd.notna(best["epoch"]) else None)

for k in ["eval_iou", "eval_dice", "eval_loss"]:
    if k in best and pd.notna(best[k]):
        print(f"{k}: {best[k]}")

print("\nCheckpoint folder:", guess_checkpoint_path(RUN_DIR, best["step"]))
print("Logged in file   :", best["_trainer_state_path"])


Metric used: ('eval_iou', 'max')

=== BEST CHECKPOINT ===
step : 8000
epoch: 3.77
eval_iou: 0.6510875225067139
eval_dice: 0.4844930171966553
eval_loss: 0.6885702013969421

Checkpoint folder: ../fine_tune/clipseg_lora_cloudsen/checkpoint-8000
Logged in file   : ../fine_tune/clipseg_lora_cloudsen/checkpoint-8000/trainer_state.json


In [10]:
import os, glob, json, re
import pandas as pd

RUN_DIR = "../fine_tune/clipseg_lora_cloudsen"

paths = sorted(glob.glob(os.path.join(RUN_DIR, "checkpoint-*", "trainer_state.json")))
print("trainer_state.json files:", len(paths))
for p in paths:
    print(" -", p)

def ckpt_step(p):
    m = re.search(r"checkpoint-(\d+)", p)
    return int(m.group(1)) if m else -1

rows = []
for p in paths:
    with open(p, "r") as f:
        st = json.load(f)
    for h in st.get("log_history", []):
        if "eval_loss" in h:
            r = dict(h)
            r["_source_ckpt_folder"] = ckpt_step(p)
            r["_source_file"] = p
            rows.append(r)

eval_df = pd.DataFrame(rows)
print("\nEval entries found:", len(eval_df))

if len(eval_df) == 0:
    raise RuntimeError("No eval entries found at all (evaluation did not run or labels were missing).")

eval_df["step"] = pd.to_numeric(eval_df["step"], errors="coerce")
eval_df["epoch"] = pd.to_numeric(eval_df["epoch"], errors="coerce")
eval_df["eval_iou"] = pd.to_numeric(eval_df.get("eval_iou"), errors="coerce")

eval_df = (eval_df.dropna(subset=["step"])
           .sort_values("step")
           .drop_duplicates("step", keep="last")
           .reset_index(drop=True))

steps = eval_df["step"].astype(int).tolist()
print("\nUnique eval steps (sorted):")
print(steps)

print("\nEval step range:", steps[0], "→", steps[-1])
print("Expected cadence (your eval_steps):", 1000)

expected = list(range(1000, steps[-1] + 1, 1000))
missing = sorted(set(expected) - set(steps))
print("\nMissing expected eval steps:", missing if missing else "None")

best = eval_df.loc[eval_df["eval_iou"].idxmax()]
print("\n=== BEST by eval_iou ===")
print("step :", int(best["step"]))
print("epoch:", float(best["epoch"]))
print("eval_iou :", float(best["eval_iou"]))
print("eval_dice:", best.get("eval_dice"))
print("eval_loss:", best.get("eval_loss"))
print("Source file:", best["_source_file"])

print("\nBest checkpoint folder (to use for inference):",
      os.path.join(RUN_DIR, f"checkpoint-{int(best['step'])}"))


trainer_state.json files: 3
 - ../fine_tune/clipseg_lora_cloudsen/checkpoint-10000/trainer_state.json
 - ../fine_tune/clipseg_lora_cloudsen/checkpoint-8000/trainer_state.json
 - ../fine_tune/clipseg_lora_cloudsen/checkpoint-9000/trainer_state.json

Eval entries found: 27

Unique eval steps (sorted):
[1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000]

Eval step range: 1000 → 10000
Expected cadence (your eval_steps): 1000

Missing expected eval steps: None

=== BEST by eval_iou ===
step : 8000
epoch: 3.77
eval_iou : 0.6510875225067139
eval_dice: 0.4844930171966553
eval_loss: 0.6885702013969421
Source file: ../fine_tune/clipseg_lora_cloudsen/checkpoint-8000/trainer_state.json

Best checkpoint folder (to use for inference): ../fine_tune/clipseg_lora_cloudsen/checkpoint-8000
